In [ ]:
# RabindricGPT — One-Cell Idempotent Colab Bootstrap & Supervisor
# Automates Drive mounting, git synchronization, artifact extraction, preflight gating, and L4 training execution.

import hashlib
import os
import sys
import shutil
from pathlib import Path

# 1. Mount Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print(f"Drive mount skipped or already mounted: {e}")

# 2. Idempotent git clone or fast-forward pull
repo_path = Path('/content/RabindricGPT')
if not repo_path.exists():
    !git clone https://github.com/ASMITSARKAR/RabindricGPT.git /content/RabindricGPT
else:
    !git -C /content/RabindricGPT pull --ff-only

os.chdir('/content/RabindricGPT')
if '/content/RabindricGPT' not in sys.path:
    sys.path.insert(0, '/content/RabindricGPT')

# 3. Setup persistent Drive folder layout and symlink runs
drive_base = Path('/content/drive/MyDrive/RabindricGPT')
drive_runs = drive_base / 'runs'
drive_prep = drive_base / 'prepared'
drive_runs.mkdir(parents=True, exist_ok=True)
drive_prep.mkdir(parents=True, exist_ok=True)

local_runs = Path('/content/RabindricGPT/runs')
if local_runs.is_symlink() or local_runs.exists():
    if local_runs.is_symlink():
        local_runs.unlink()
    else:
        shutil.rmtree(local_runs)
os.symlink(str(drive_runs), str(local_runs))

# 4. Extract or refresh shards and tokenizers from Drive when Drive hash differs
def compute_file_sha(p: Path) -> str:
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

for archive in ['tok.zip', 'shards.zip']:
    src = drive_prep / archive if (drive_prep / archive).exists() else (drive_base / archive)
    folder = archive.replace('.zip', '')
    local_folder = Path('/content/RabindricGPT') / folder
    hash_stamp_file = local_folder / '.drive_hash'

    if src.exists():
        drive_hash = compute_file_sha(src)
        local_hash = hash_stamp_file.read_text(encoding='utf-8').strip() if hash_stamp_file.exists() else ''
        if (not local_folder.exists()) or (drive_hash != local_hash):
            print(f"Refreshing {folder} from {src} (Drive SHA: {drive_hash[:12]} vs Local: {local_hash[:12] or 'none'})...")
            !unzip -q -o "{src}" -d /content/RabindricGPT/
            hash_stamp_file.parent.mkdir(parents=True, exist_ok=True)
            hash_stamp_file.write_text(drive_hash, encoding='utf-8')
        else:
            print(f"{folder} is up to date with Drive (SHA: {drive_hash[:12]}). Skipping unpack.")

# 5. Install dependencies
!pip install -q sentencepiece pyyaml

# 6. Run quick CPU preflight gating
!python scripts/00_preflight.py --quick

# 7. Launch supervised queue runner on L4 GPU
!python colab/runner_lib.py --queue configs/runs/queue.yaml